# Build `kurucz_cd23_chianti_H_He_latest.h5`

This notebook builds a TARDIS atom-data file from the current Carsus public data inputs:

- NIST atomic weights and ionization energies for H-Zn
- Kurucz CD23 GFALL levels and lines from the [Carsus Kurucz data mirror](https://github.com/tardis-sn/carsus-data-kurucz)
- CHIANTI H-He levels, lines, and electron collision strengths from the [CHIANTI database](https://www.chiantidatabase.org/) configured by `XUVTOP`
- Knox-Long recombination zeta data
- [NNDC ENSDF CSV decay-radiation data](https://github.com/tardis-sn/carsus-data-nndc)

The output uses the legacy TARDIS HDF schema (`database_version='v0.9'`) expected by TARDIS regression atom-data files. It is intended to make a valid current-data file named `kurucz_cd23_chianti_H_He_latest.h5`; it is not a byte-for-byte reproduction of older historical regression files.

## Configure source locations

Download and extract the latest CHIANTI database from the CHIANTI project, then set `XUVTOP` to the extracted database root before running the Carsus readers. The current CHIANTI database package should contain files such as `VERSION`, `masterlist/masterlist_ions.pkl`, `h/h_1/h_1.elvlc`, and `he/he_1/he_1.elvlc`.

The optional `CHIANTI_DATABASE_URL` block below is provided for reproducible local runs when you know the exact upstream archive URL. If `XUVTOP` already points to an extracted CHIANTI tree, the notebook will use that tree and skip downloading.

In [ ]:
from pathlib import Path
import os
import shutil
import tarfile
import urllib.request

DATA_ROOT = Path(os.environ.get("CARSUS_ATOMDATA_SOURCE_DIR", Path.home() / "Downloads" / "carsus-atomdata-sources"))
DATA_ROOT.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = Path("kurucz_cd23_chianti_H_He_latest.h5")

# Prefer an existing XUVTOP. If it is not set, use DATA_ROOT/chianti after optional download/extraction.
CHIANTI_ROOT = Path(os.environ.get("XUVTOP", DATA_ROOT / "chianti")).expanduser()
CHIANTI_DATABASE_URL = os.environ.get("CHIANTI_DATABASE_URL", "")

if not CHIANTI_ROOT.exists() and CHIANTI_DATABASE_URL:
    archive_path = DATA_ROOT / Path(CHIANTI_DATABASE_URL).name
    print(f"Downloading CHIANTI database archive to {archive_path}")
    urllib.request.urlretrieve(CHIANTI_DATABASE_URL, archive_path)
    extract_root = DATA_ROOT / "chianti-extracted"
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir()
    with tarfile.open(archive_path) as archive:
        archive.extractall(extract_root)
    candidates = [path for path in extract_root.rglob("VERSION")]
    if not candidates:
        raise RuntimeError("The CHIANTI archive did not contain a VERSION file.")
    CHIANTI_ROOT = candidates[0].parent

required_chianti_paths = [
    CHIANTI_ROOT / "VERSION",
    CHIANTI_ROOT / "masterlist" / "masterlist_ions.pkl",
    CHIANTI_ROOT / "h" / "h_1" / "h_1.elvlc",
    CHIANTI_ROOT / "he" / "he_1" / "he_1.elvlc",
    CHIANTI_ROOT / "he" / "he_2" / "he_2.elvlc",
]
missing = [path for path in required_chianti_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Set XUVTOP to an extracted CHIANTI database root or set CHIANTI_DATABASE_URL. "
        f"Missing: {missing}"
    )

os.environ["XUVTOP"] = str(CHIANTI_ROOT)
print(f"Using CHIANTI database: {CHIANTI_ROOT}")
print((CHIANTI_ROOT / "VERSION").read_text().strip())

## Create Carsus readers

Import `carsus` before importing `astropy.units` or `astropy.constants` elsewhere in the kernel. Carsus pins the Astropy constants set used by legacy TARDIS atom-data generation during import.

In [ ]:
from carsus.io.chianti_ import ChiantiReader
from carsus.io.kurucz import GFALLReader
from carsus.io.nist import NISTIonizationEnergies, NISTWeightsComp
from carsus.io.nuclear import NNDCReader
from carsus.io.zeta import KnoxLongZeta

atomic_weights = NISTWeightsComp()
ionization_energies = NISTIonizationEnergies("H-Zn")

# GFALLReader's default source is the Carsus Kurucz CD23 mirror on the main branch.
# Include labels in the level identity so same-energy/J terms remain distinct when labels differ.
gfall_reader = GFALLReader(
    "H-Zn",
    unique_level_identifier=["energy", "j", "label"],
)

chianti_reader = ChiantiReader("H-He", collisions=True, priority=20)
zeta_data = KnoxLongZeta()

# With remote=True, NNDCReader clones the Carsus NNDC CSV repository into its
# default location if it is not already present.
nndc_reader = NNDCReader(remote=True)

## Build and write the TARDIS atom-data file

In [ ]:
from carsus.io.output import TARDISAtomData

atom_data = TARDISAtomData(
    atomic_weights=atomic_weights,
    ionization_energies=ionization_energies,
    gfall_reader=gfall_reader,
    zeta_data=zeta_data,
    chianti_reader=chianti_reader,
    nndc_reader=nndc_reader,
)

atom_data.to_hdf(
    OUTPUT_PATH,
    legacy_tardis_schema=True,
    database_version="v0.9",
)
print(f"Wrote {OUTPUT_PATH.resolve()}")

## Validate the file layout

These checks verify the legacy TARDIS schema contract used by the Carsus regression test for `kurucz_cd23_chianti_H_He_latest.h5`.

In [ ]:
import pandas as pd

expected_keys = {
    "/atom_data",
    "/collisions_data",
    "/collisions_metadata",
    "/decay_radiation_data",
    "/ionization_data",
    "/levels_data",
    "/lines_data",
    "/macro_atom_data",
    "/macro_atom_references",
    "/metadata",
    "/zeta_data",
}

with pd.HDFStore(OUTPUT_PATH, mode="r") as store:
    keys = set(store.keys())
    assert keys == expected_keys, sorted(keys ^ expected_keys)
    assert store.root._v_attrs["FORMAT_VERSION"] == "2.0"
    assert store.root._v_attrs["database_version"] == "v0.9"

    metadata = store["metadata"]
    assert ("md5sum", "levels") in metadata.index
    assert ("md5sum", "lines") in metadata.index
    assert ("md5sum", "levels_data") not in metadata.index
    assert ("md5sum", "lines_data") not in metadata.index

    atom_data_table = store["atom_data"]
    assert atom_data_table.index.name == "atomic_number"
    assert atom_data_table.index.min() == 1
    assert atom_data_table.index.max() == 94

    print("Validated legacy TARDIS atom-data schema.")
    print("Rows:")
    for key in sorted(keys):
        obj = store[key]
        shape = getattr(obj, "shape", None)
        print(f"  {key}: {shape}")

## Record source versions

The file metadata stores checksums and software versions. The printed values below are useful when publishing or uploading the generated atom-data file.

In [ ]:
with pd.HDFStore(OUTPUT_PATH, mode="r") as store:
    print("HDF root attributes:")
    for name in ["FORMAT_VERSION", "database_version", "DATE", "MD5", "UUID1"]:
        print(f"  {name}: {store.root._v_attrs[name]}")

    print("\nMetadata:")
    display(store["metadata"])

print(f"CHIANTI XUVTOP: {os.environ['XUVTOP']}")
print(f"CHIANTI version: {(Path(os.environ['XUVTOP']) / 'VERSION').read_text().strip()}")
print(f"NNDC source directory: {nndc_reader.dirname}")
print(f"GFALL checksum: {gfall_reader.version}")